In [ ]:
# ── Cell 1 — Config ───────────────────────────────────────────────────────────

panels = {
    "CGN":   ["C3","CD46","CFH","CFHR5","CFI","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["ACE","AGT","AGTR1","BMP4","CHD1L","CHRM3","DSTYK","EYA1","FGF20","FRAS1",
              "FREM1","FREM2","GATA3","GRIP1","HNF1B","HPSE2","ITGA8","KAL1","LRIG2","MUC1",
              "PAX2","REN","RET","ROBO2","SALL1","SIX1","SIX2","SIX5","SOX17","SRGAP1",
              "TBX18","TNXB","TRAP1","UMOD","UPK3A","WNT4"],
    "SRNS":  ["ACTN4","ADCK4","ANLN","ARHGAP24","ARHGDIA","CD2AP","COQ2","COQ6","CRB2",
          "CUBN","DGKE","EMP2","FAT1","INF2","ITGA3","ITGB4","KANK1","KANK2","KANK4",
          "LAMB2","LMX1B","MTTL1","MYH9","MYO1E","NPHS1","NPHS2","NUP107","NUP205",
          "NUP93","PDSS2","PLCE1","PTPRO","SCARB2","SMARCAL1","TRPC6","WDR73","WT1","XPO5"],
    "USD":   ["ADCY10","AGXT","APRT","ATP6V0A4","ATP6V1B1","CA2","CASR","CLCN5","CLCNKB",
              "CLDN16","CLDN19","CYP24A1","FAM20A","GRHPR","HNF4A","HOGA1","HPRT1","KCNJ1",
              "OCRL","SLC12A1","SLC22A12","SLC2A9","SLC34A1","SLC34A3","SLC3A1","SLC4A1",
              "SLC7A9","SLC9A3R1","VDR","XDH"],
    "NPHP":  ["ANKS6","CEP164","CEP290","GLIS2","INVS","IQCB1","NEK8","NPHP1","NPHP3",
              "NPHP4","RPGRIP1L","SDCCAG8","TMEM67","TTC21B","WDR19","ZNF423"],
}

gene_panel   = {gene: panel for panel, genes in panels.items() for gene in genes}
all_genes    = list(gene_panel.keys())

rsa_base     = "https://raw.githubusercontent.com/NephVar/NephVar/main/biophysical"
clinvar_base = "https://raw.githubusercontent.com/Joshua-Pillai/NephVar/main"

buried_threshold = 0.25  # RSA cutoff for buried vs exposed (Tien et al. 2013 framework)

# ACMG classification buckets — includes low-penetrance composite labels,
# consistent with the IDR notebook's bucket() so P/LP and B/LB counts match
# across analyses in the same manuscript.
PATHOGENIC = {"Pathogenic","Likely pathogenic","Pathogenic/Likely pathogenic",
              "Pathogenic, low penetrance","Likely pathogenic, low penetrance",
              "Pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Pathogenic, low penetrance"}
VUS_SET    = {"Uncertain significance","Uncertain significance/Uncertain risk allele",
              "Uncertain risk allele","Likely pathogenic/Likely risk allele",
              "Likely risk allele","Uncertain significance/VUS-mid"}
BENIGN     = {"Benign","Likely benign","Benign/Likely benign"}

def bucket(label):
    if pd.isna(label):
        return "Other"
    label = str(label).strip()

    # Exact-match sets first (fast path for the common cases)
    if label in PATHOGENIC:
        return "P" if label == "Pathogenic" else "LP"
    if label in VUS_SET:
        return "VUS"
    if label in BENIGN:
        return "B" if label == "Benign" else "LB"

    # Composite/qualifier forms: "Pathogenic; other", "Likely benign; risk factor"
    base = label.split(";")[0].strip()
    if base == "Pathogenic":         return "P"
    if base == "Likely pathogenic":  return "LP"
    if base == "Benign":             return "B"
    if base == "Likely benign":      return "LB"

    # VUS subtiers: "VUS-low", "VUS-mid", "VUS-high", "Uncertain significance/VUS-..."
    if "VUS" in label or "Uncertain significance" in label:
        return "VUS"

    return "Other"

In [ ]:
# ── Cell 2 — Load RSA ─────────────────────────────────────────────────────────
import re, json, urllib.request, pandas as pd

rows = []
errors = []

for gene in all_genes:
    url = f"{rsa_base}/{gene}.html"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            html = r.read().decode("utf-8", errors="replace")
        m = re.search(r'const residues = (\[.*?\]);', html, re.DOTALL)
        if not m:
            raise ValueError("residues block not found")
        residues = json.loads(m.group(1))
        for res in residues:
            rows.append({
                "gene"  : gene,
                "resnum": res["resnum"],
                "rsa"   : res["rsa"],
                "ss"    : res["ss"],
                "plddt" : res["plddt"],
            })
    except Exception as e:
        errors.append(gene)
        print(f"  ERROR {gene}: {e}")

df_rsa = pd.DataFrame(rows)

print(f"Genes loaded : {df_rsa['gene'].nunique()} / {len(all_genes)}")
print(f"Total residues: {len(df_rsa):,}")
if errors:
    print(f"Errors       : {errors}")

  ERROR MTTL1: HTTP Error 404: Not Found
Genes loaded : 129 / 130
Total residues: 127,472
Errors       : ['MTTL1']


In [ ]:
# ── Cell 3 — Load ClinVar missense ────────────────────────────────────────────
import io, re, urllib.request
import pandas as pd

# Build a fast RSA lookup set: (gene, resnum) pairs that actually exist in df_rsa
rsa_residues = set(zip(df_rsa['gene'], df_rsa['resnum']))

def parse_resnum(protein_change, gene):
    """
    Try each comma-separated protein change entry in order.
    Return the first resnum that exists in the RSA table for this gene.
    Falls back to the first parseable resnum if none match (e.g. gene has no RSA).
    """
    if pd.isna(protein_change):
        return None
    first_parsed = None
    for part in str(protein_change).split(','):
        part = part.strip()
        m = re.search(r'[A-Za-z*](\d+)', part)
        if m:
            resnum = int(m.group(1))
            if first_parsed is None:
                first_parsed = resnum          # save first parseable as fallback
            if (gene, resnum) in rsa_residues: # prefer RSA-validated resnum
                return resnum
    return first_parsed                        # fallback: first parseable if no RSA match found

rows = []
errors = []

for panel, genes in panels.items():
    for gene in genes:
        url = f"{clinvar_base}/{panel}/{gene}.txt"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                content = r.read().decode("utf-8", errors="replace")
            df = pd.read_csv(io.StringIO(content), sep="\t", low_memory=False)
            df["gene"]  = gene
            df["panel"] = panel
            rows.append(df)
        except Exception as e:
            errors.append(gene)
            print(f"  ERROR {gene}: {e}")

df_raw = pd.concat(rows, ignore_index=True)
n_raw = len(df_raw)

# ── Global VariationID dedup — collapse multi-gene CNVs/structural variants
n_dup_rows = df_raw['VariationID'].duplicated(keep='first').sum()
df_raw = df_raw.drop_duplicates(subset='VariationID', keep='first').copy()
print(f"Raw rows: {n_raw:,}  |  Removed {n_dup_rows:,} duplicate VariationIDs  |  Unique: {len(df_raw):,}")
assert len(df_raw) == 117373, f"Expected 117,373 unique variants, got {len(df_raw):,}"

# ── Filter missense
df_miss = df_raw[
    df_raw["Molecular consequence"].str.contains("missense", case=False, na=False)
].copy()

# ── Parse resnum — RSA-validated, one row per variant (no expansion/duplication)
df_miss["resnum"] = df_miss.apply(
    lambda row: parse_resnum(row["Protein change"], row["gene"]), axis=1
)

# ── ACMG bucket
df_miss["acmg"] = df_miss["Germline classification"].apply(bucket)

# ── Report
n_total    = len(df_miss)
n_parsed   = df_miss["resnum"].notna().sum()
n_unparsed = n_total - n_parsed

print(f"Total missense   : {n_total:,}")
print(f"Residue parsed   : {n_parsed:,} ({100*n_parsed/n_total:.1f}%)")
print(f"Unparseable      : {n_unparsed}")
print(f"\nACMG distribution:")
print(df_miss["acmg"].value_counts().to_string())
if errors:
    print(f"\nErrors: {errors}")

Raw rows: 119,315  |  Removed 1,942 duplicate VariationIDs  |  Unique: 117,373
Total missense   : 50,462
Residue parsed   : 50,425 (99.9%)
Unparseable      : 37

ACMG distribution:
acmg
VUS    44113
LP      2616
LB      2241
P        899
B        593


In [ ]:
# ── Cell 4 — Join RSA to missense variants ────────────────────────────────────

# Drop unparseable residues and Other ACMG
df_miss_clean = df_miss[
    df_miss["resnum"].notna() &
    (df_miss["acmg"] != "Other")
].copy()

# Merge on gene + resnum
df_merged = df_miss_clean.merge(
    df_rsa[["gene", "resnum", "rsa", "ss", "plddt"]],
    on=["gene", "resnum"],
    how="left"
)

# Check RSA match rate
n_total    = len(df_merged)
n_rsa      = df_merged["rsa"].notna().sum()
n_no_rsa   = n_total - n_rsa

print(f"Variants into join  : {n_total:,}")
print(f"RSA matched         : {n_rsa:,} ({100*n_rsa/n_total:.1f}%)")
print(f"No RSA match        : {n_no_rsa:,}")
print(f"\nACMG counts after join:")
print(df_merged[df_merged["rsa"].notna()]["acmg"].value_counts().to_string())
print(f"\nSample rows:")
print(df_merged[df_merged["rsa"].notna()][["gene","resnum","acmg","rsa","ss","plddt"]].head(8).to_string(index=False))

Variants into join  : 50,425
RSA matched         : 50,346 (99.8%)
No RSA match        : 79

ACMG counts after join:
acmg
VUS    44021
LP      2609
LB      2230
P        895
B        591

Sample rows:
gene  resnum acmg    rsa    ss  plddt
  C3   336.0  VUS 0.2065  loop  47.44
  C3   339.0  VUS 0.0747 sheet  65.50
  C3   341.0  VUS 0.1395 sheet  72.44
  C3   343.0  VUS 0.3978 sheet  82.75
  C3   345.0  VUS 0.3654 sheet  79.50
  C3   346.0  VUS 0.0152 sheet  87.56
  C3   354.0  VUS 0.3378 sheet  89.06
  C3   361.0  VUS 0.1447  loop  88.44


In [ ]:
# ── Cell 5 — Summary statistics ───────────────────────────────────────────────
import numpy as np

df_rsa_clean = df_merged[df_merged["rsa"].notna()].copy()

order = ["P", "LP", "VUS", "LB", "B"]

rows = []
for acmg in order:
    sub = df_rsa_clean[df_rsa_clean["acmg"] == acmg]["rsa"]
    rows.append({
        "acmg"      : acmg,
        "n"         : len(sub),
        "median_rsa": round(sub.median(), 4),
        "mean_rsa"  : round(sub.mean(), 4),
        "q25"       : round(sub.quantile(0.25), 4),
        "q75"       : round(sub.quantile(0.75), 4),
        "pct_buried": round(100 * (sub < 0.25).sum() / len(sub), 1),
    })

df_summary = pd.DataFrame(rows)

print("RSA summary by ACMG category")
print("=" * 65)
print(f"{'acmg':<6} {'n':>7} {'median':>8} {'mean':>8} {'Q25':>8} {'Q75':>8} {'%buried':>8}")
print("-" * 65)
for _, r in df_summary.iterrows():
    print(f"{r['acmg']:<6} {int(r['n']):>7} {r['median_rsa']:>8.4f} {r['mean_rsa']:>8.4f} "
          f"{r['q25']:>8.4f} {r['q75']:>8.4f} {r['pct_buried']:>7.1f}%")

    # ── Save pooled source data by ACMG category for GraphPad ────────────────────

cols = ["VariationID", "gene", "panel", "acmg", "rsa", "ss", "plddt"]

for acmg in order:
    sub = df_rsa_clean[df_rsa_clean["acmg"] == acmg][cols]
    fname = f"rsa_pooled_{acmg}.csv"
    sub.to_csv(fname, index=False)
    print(f"Saved: {fname} ({len(sub):,} rows)")

RSA summary by ACMG category
acmg         n   median     mean      Q25      Q75  %buried
-----------------------------------------------------------------
P          895   0.2537   0.3340   0.0240   0.6250    49.9%
LP        2609   0.4423   0.3981   0.0769   0.6538    37.9%
VUS      44021   0.4000   0.4013   0.1417   0.6513    35.2%
LB        2230   0.5138   0.4812   0.2500   0.7263    24.9%
B          591   0.4826   0.4652   0.2409   0.7040    25.9%
Saved: rsa_pooled_P.csv (895 rows)
Saved: rsa_pooled_LP.csv (2,609 rows)
Saved: rsa_pooled_VUS.csv (44,021 rows)
Saved: rsa_pooled_LB.csv (2,230 rows)
Saved: rsa_pooled_B.csv (591 rows)


In [ ]:
# ── Cell 5b — Summary statistics by panel ─────────────────────────────────────

order = ["P", "LP", "VUS", "LB", "B"]

rows = []
for panel in ["CGN", "CAKUT", "SRNS", "USD", "NPHP"]:
    sub_panel = df_rsa_clean[df_rsa_clean["panel"] == panel]
    for acmg in order:
        sub = sub_panel[sub_panel["acmg"] == acmg]["rsa"]
        if len(sub) == 0:
            continue
        rows.append({
            "panel"     : panel,
            "acmg"      : acmg,
            "n"         : len(sub),
            "median_rsa": round(sub.median(), 4),
            "mean_rsa"  : round(sub.mean(), 4),
            "q25"       : round(sub.quantile(0.25), 4),
            "q75"       : round(sub.quantile(0.75), 4),
            "pct_buried": round(100 * (sub < 0.25).sum() / len(sub), 1),
        })

df_summary_panel = pd.DataFrame(rows)

for panel in ["CGN", "CAKUT", "SRNS", "USD", "NPHP"]:
    sub = df_summary_panel[df_summary_panel["panel"] == panel]
    print(f"\n{panel}")
    print(f"  {'acmg':<6} {'n':>6} {'median':>8} {'mean':>8} {'Q25':>8} {'Q75':>8} {'%buried':>8}")
    print("  " + "-" * 58)
    for _, r in sub.iterrows():
        print(f"  {r['acmg']:<6} {int(r['n']):>6} {r['median_rsa']:>8.4f} {r['mean_rsa']:>8.4f} "
              f"{r['q25']:>8.4f} {r['q75']:>8.4f} {r['pct_buried']:>7.1f}%")


CGN
  acmg        n   median     mean      Q25      Q75  %buried
  ----------------------------------------------------------
  P         256   0.6442   0.5868   0.5769   0.7019    10.5%
  LP       1240   0.6346   0.5606   0.5361   0.6923    13.8%
  VUS      5189   0.4615   0.4514   0.1875   0.7287    30.7%
  LB        343   0.7161   0.5942   0.4128   0.8174    16.9%
  B          59   0.7442   0.6512   0.4913   0.8381     6.8%

CAKUT
  acmg        n   median     mean      Q25      Q75  %buried
  ----------------------------------------------------------
  P         148   0.1943   0.2561   0.0096   0.3623    58.8%
  LP        294   0.2347   0.3017   0.0520   0.4828    51.4%
  VUS     11984   0.3814   0.3955   0.1447   0.6457    36.0%
  LB        629   0.4619   0.4574   0.2372   0.7111    26.4%
  B         182   0.3975   0.4121   0.1935   0.6416    31.3%

SRNS
  acmg        n   median     mean      Q25      Q75  %buried
  ----------------------------------------------------------
  P   

In [ ]:
# ── Save per-category per-panel source files for GraphPad ─────────────────────

cols = ["VariationID", "gene", "panel", "acmg", "rsa", "ss", "plddt"]
order  = ["B", "LB", "VUS", "LP", "P"]
panels = ["CGN", "CAKUT", "SRNS", "USD", "NPHP"]

for panel in panels:
    for acmg in order:
        sub = df_rsa_clean[
            (df_rsa_clean["panel"] == panel) &
            (df_rsa_clean["acmg"]  == acmg)
        ][cols]
        fname = f"rsa_{panel}_{acmg}.csv"
        sub.to_csv(fname, index=False)
        print(f"Saved: {fname} ({len(sub):,} rows)")

Saved: rsa_CGN_B.csv (59 rows)
Saved: rsa_CGN_LB.csv (343 rows)
Saved: rsa_CGN_VUS.csv (5,189 rows)
Saved: rsa_CGN_LP.csv (1,240 rows)
Saved: rsa_CGN_P.csv (256 rows)
Saved: rsa_CAKUT_B.csv (182 rows)
Saved: rsa_CAKUT_LB.csv (629 rows)
Saved: rsa_CAKUT_VUS.csv (11,984 rows)
Saved: rsa_CAKUT_LP.csv (294 rows)
Saved: rsa_CAKUT_P.csv (148 rows)
Saved: rsa_SRNS_B.csv (211 rows)
Saved: rsa_SRNS_LB.csv (824 rows)
Saved: rsa_SRNS_VUS.csv (11,787 rows)
Saved: rsa_SRNS_LP.csv (362 rows)
Saved: rsa_SRNS_P.csv (144 rows)
Saved: rsa_USD_B.csv (78 rows)
Saved: rsa_USD_LB.csv (278 rows)
Saved: rsa_USD_VUS.csv (6,955 rows)
Saved: rsa_USD_LP.csv (596 rows)
Saved: rsa_USD_P.csv (304 rows)
Saved: rsa_NPHP_B.csv (61 rows)
Saved: rsa_NPHP_LB.csv (156 rows)
Saved: rsa_NPHP_VUS.csv (8,106 rows)
Saved: rsa_NPHP_LP.csv (117 rows)
Saved: rsa_NPHP_P.csv (43 rows)


In [ ]:
# ── Save per-panel GraphPad files (one column per ACMG category) ──────────────

order  = ["B", "LB", "VUS", "LP", "P"]
panels = ["CGN", "CAKUT", "SRNS", "USD", "NPHP"]

for panel in panels:
    sub_panel = df_rsa_clean[df_rsa_clean["panel"] == panel]

    # Build one column per ACMG category (RSA values, unequal lengths → pad with NaN)
    col_dict = {}
    for acmg in order:
        vals = sub_panel[sub_panel["acmg"] == acmg]["rsa"].values
        col_dict[acmg] = vals

    max_len = max(len(v) for v in col_dict.values())
    df_gp = pd.DataFrame({
        acmg: pd.Series(vals) for acmg, vals in col_dict.items()
    })  # pandas handles unequal lengths with NaN padding automatically

    fname = f"rsa_{panel}_graphpad.csv"
    df_gp.to_csv(fname, index=False)
    print(f"Saved: {fname}  shapes: { {a: col_dict[a].shape[0] for a in order} }")

Saved: rsa_CGN_graphpad.csv  shapes: {'B': 59, 'LB': 343, 'VUS': 5189, 'LP': 1240, 'P': 256}
Saved: rsa_CAKUT_graphpad.csv  shapes: {'B': 182, 'LB': 629, 'VUS': 11984, 'LP': 294, 'P': 148}
Saved: rsa_SRNS_graphpad.csv  shapes: {'B': 211, 'LB': 824, 'VUS': 11787, 'LP': 362, 'P': 144}
Saved: rsa_USD_graphpad.csv  shapes: {'B': 78, 'LB': 278, 'VUS': 6955, 'LP': 596, 'P': 304}
Saved: rsa_NPHP_graphpad.csv  shapes: {'B': 61, 'LB': 156, 'VUS': 8106, 'LP': 117, 'P': 43}


In [ ]:
# ── Cell 7b — Per-gene distribution table (B→P order) ────────────────────────

order = ["B", "LB", "VUS", "LP", "P"]

# Rebuild df_gene
rows = []
for gene in all_genes:
    sub_gene = df_rsa_clean[df_rsa_clean["gene"] == gene]
    panel = gene_panel[gene]
    for acmg in order:
        sub = sub_gene[sub_gene["acmg"] == acmg]["rsa"]
        rows.append({
            "gene"      : gene,
            "panel"     : panel,
            "acmg"      : acmg,
            "n"         : len(sub),
            "median_rsa": round(sub.median(), 4) if len(sub) > 0 else None,
            "mean_rsa"  : round(sub.mean(), 4)   if len(sub) > 0 else None,
            "q25"       : round(sub.quantile(0.25), 4) if len(sub) > 0 else None,
            "q75"       : round(sub.quantile(0.75), 4) if len(sub) > 0 else None,
            "pct_buried": round(100 * (sub < 0.25).sum() / len(sub), 1) if len(sub) > 0 else None,
        })

df_gene = pd.DataFrame(rows)

# Compute plp gene list
plp = df_gene[df_gene["acmg"].isin(["P", "LP"])].groupby("gene")["n"].sum().reset_index()
plp.columns = ["gene", "n_plp"]
plp = plp[plp["n_plp"] >= 10].sort_values("n_plp", ascending=False)
plp_genes = plp["gene"].tolist()

print(f"{'gene':<12} {'panel':<6} {'B':>5} {'LB':>5} {'VUS':>6} {'LP':>5} {'P':>5}  {'med_P':>7} {'%bur_P':>7}")
print("-" * 70)

for gene in plp_genes:
    sub_gene = df_rsa_clean[df_rsa_clean["gene"] == gene]
    panel    = gene_panel[gene]
    counts   = {a: len(sub_gene[sub_gene["acmg"] == a]) for a in order}
    p_rsa    = sub_gene[sub_gene["acmg"] == "P"]["rsa"]
    med_p    = f"{p_rsa.median():.3f}" if len(p_rsa) > 0 else "—"
    bur_p    = f"{100*(p_rsa < 0.25).sum()/len(p_rsa):.1f}%" if len(p_rsa) > 0 else "—"
    print(f"{gene:<12} {panel:<6} {counts['B']:>5} {counts['LB']:>5} "
          f"{counts['VUS']:>6} {counts['LP']:>5} {counts['P']:>5}  {med_p:>7} {bur_p:>7}")

df_gene.to_csv("rsa_summary_by_gene.csv", index=False)
print(f"\nSaved: rsa_summary_by_gene.csv")

gene         panel      B    LB    VUS    LP     P    med_P  %bur_P
----------------------------------------------------------------------
COL4A5       CGN       17    95    429   508   203    0.644    5.4%
COL4A3       CGN       11    45    615   326    22    0.673    4.5%
COL4A4       CGN        8    42    702   268    16    0.587   18.8%
AGXT         USD        2    12    142    88    49    0.035   81.6%
CASR         USD        3    14   1470    88    48    0.231   54.2%
UMOD         CAKUT      2    13    249    72    29    0.162   58.6%
NPHS1        SRNS       3    36    385    72    16    0.006   81.2%
RET          CAKUT      0    55   1856    38    45    0.017   88.9%
HNF1B        CAKUT      2     7    260    64    17    0.191   52.9%
HNF4A        USD        6     5    208    50    21    0.102   71.4%
TMEM67       NPHP       2     9    419    50    21    0.161   61.9%
CFI          CGN        2    15    331    54     4    0.063  100.0%
HOGA1        USD        0     5    123    46 

In [ ]:
# ── Cell 5c — ACMG × Secondary structure cross-tab ───────────────────────────

order  = ["P", "LP", "VUS", "LB", "B"]
ss_cats = ["helix", "sheet", "loop"]

# ── Pooled ────────────────────────────────────────────────────────────────────
print("POOLED — SS distribution by ACMG category")
print("=" * 65)
print(f"{'acmg':<6} {'n':>7}  {'helix':>8} {'sheet':>8} {'loop':>8}  {'med_RSA':>8}")
print("-" * 65)

rows = []
for acmg in order:
    sub = df_rsa_clean[df_rsa_clean["acmg"] == acmg]
    n   = len(sub)
    ss_counts = sub["ss"].value_counts()
    h = ss_counts.get("helix", 0)
    s = ss_counts.get("sheet", 0)
    l = ss_counts.get("loop",  0)
    med = sub["rsa"].median()
    print(f"{acmg:<6} {n:>7}  {100*h/n:>7.1f}% {100*s/n:>7.1f}% {100*l/n:>7.1f}%  {med:>8.4f}")
    rows.append({"acmg": acmg, "n": n,
                 "pct_helix": round(100*h/n,1),
                 "pct_sheet": round(100*s/n,1),
                 "pct_loop" : round(100*l/n,1),
                 "median_rsa": round(med,4)})

df_ss_pooled = pd.DataFrame(rows)

# ── Per panel ─────────────────────────────────────────────────────────────────
panel_rows = []
for panel in ["CGN", "CAKUT", "SRNS", "USD", "NPHP"]:
    print(f"\n{panel}")
    print(f"  {'acmg':<6} {'n':>6}  {'helix':>8} {'sheet':>8} {'loop':>8}  {'med_RSA':>8}")
    print("  " + "-" * 55)
    sub_panel = df_rsa_clean[df_rsa_clean["panel"] == panel]
    for acmg in order:
        sub = sub_panel[sub_panel["acmg"] == acmg]
        n   = len(sub)
        if n == 0:
            continue
        ss_counts = sub["ss"].value_counts()
        h = ss_counts.get("helix", 0)
        s = ss_counts.get("sheet", 0)
        l = ss_counts.get("loop",  0)
        med = sub["rsa"].median()
        print(f"  {acmg:<6} {n:>6}  {100*h/n:>7.1f}% {100*s/n:>7.1f}% {100*l/n:>7.1f}%  {med:>8.4f}")
        panel_rows.append({"panel": panel, "acmg": acmg, "n": n,
                           "pct_helix": round(100*h/n,1),
                           "pct_sheet": round(100*s/n,1),
                           "pct_loop" : round(100*l/n,1),
                           "median_rsa": round(med,4)})

df_ss_panel = pd.DataFrame(panel_rows)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ss_pooled.to_csv("ss_summary_pooled.csv", index=False)
df_ss_panel.to_csv("ss_summary_by_panel.csv",  index=False)
print(f"\nSaved: ss_summary_pooled.csv")
print(f"Saved: ss_summary_by_panel.csv")

POOLED — SS distribution by ACMG category
acmg         n     helix    sheet     loop   med_RSA
-----------------------------------------------------------------
P          895     27.5%    18.4%    54.1%    0.2537
LP        2609     21.4%    13.5%    65.1%    0.4423
VUS      44021     28.0%    19.7%    52.3%    0.4000
LB        2230     21.2%    19.7%    59.1%    0.5138
B          591     23.0%    20.1%    56.9%    0.4826

CGN
  acmg        n     helix    sheet     loop   med_RSA
  -------------------------------------------------------
  P         256      0.4%     7.4%    92.2%    0.6442
  LP       1240      1.1%     7.3%    91.6%    0.6346
  VUS      5189      5.9%    26.2%    67.8%    0.4615
  LB        343      2.0%    20.1%    77.8%    0.7161
  B          59      0.0%    13.6%    86.4%    0.7442

CAKUT
  acmg        n     helix    sheet     loop   med_RSA
  -------------------------------------------------------
  P         148     30.4%    31.8%    37.8%    0.1943
  LP        29

In [ ]:
# ── Cell 8b — Surface-exposed P/LP excluding CGN ─────────────────────────────

for acmg in ["P", "LP"]:
    sub = df_rsa_clean[
        (df_rsa_clean["acmg"]  == acmg) &
        (df_rsa_clean["panel"] != "CGN")
    ].copy()
    buried  = sub[sub["rsa"] <  buried_threshold]
    exposed = sub[sub["rsa"] >= buried_threshold]

    print(f"{'═'*65}")
    print(f"  {acmg}  —  non-CGN only  (buried vs exposed)")
    print(f"{'═'*65}")
    print(f"  total : {len(sub):>5}")
    print(f"  buried: {len(buried):>5} ({100*len(buried)/len(sub):.1f}%)  "
          f"median RSA {buried['rsa'].median():.4f}  "
          f"median pLDDT {buried['plddt'].median():.1f}")
    print(f"  exposed:{len(exposed):>5} ({100*len(exposed)/len(sub):.1f}%)  "
          f"median RSA {exposed['rsa'].median():.4f}  "
          f"median pLDDT {exposed['plddt'].median():.1f}")

    print(f"\n  Exposed {acmg} by panel (non-CGN):")
    print(f"  {'panel':<8} {'n_exp':>6} {'%exp':>6}  {'med_RSA':>8}  {'med_pLDDT':>10}")
    print(f"  {'-'*45}")
    for panel in ["CAKUT","SRNS","USD","NPHP"]:
        sub_p = sub[sub["panel"] == panel]
        exp_p = sub_p[sub_p["rsa"] >= buried_threshold]
        if len(sub_p) == 0: continue
        print(f"  {panel:<8} {len(exp_p):>6} {100*len(exp_p)/len(sub_p):>5.1f}%  "
              f"{exp_p['rsa'].median():>8.4f}  "
              f"{exp_p['plddt'].median():>10.1f}")
    print()

═════════════════════════════════════════════════════════════════
  P  —  non-CGN only  (buried vs exposed)
═════════════════════════════════════════════════════════════════
  total :   639
  buried:   420 (65.7%)  median RSA 0.0240  median pLDDT 95.1
  exposed:  219 (34.3%)  median RSA 0.5333  median pLDDT 85.9

  Exposed P by panel (non-CGN):
  panel     n_exp   %exp   med_RSA   med_pLDDT
  ---------------------------------------------
  CAKUT        61  41.2%    0.4519        92.4
  SRNS         60  41.7%    0.5583        75.8
  USD          84  27.6%    0.5424        88.2
  NPHP         14  32.6%    0.4267        83.4

═════════════════════════════════════════════════════════════════
  LP  —  non-CGN only  (buried vs exposed)
═════════════════════════════════════════════════════════════════
  total :  1369
  buried:   819 (59.8%)  median RSA 0.0328  median pLDDT 93.9
  exposed:  550 (40.2%)  median RSA 0.4710  median pLDDT 87.3

  Exposed LP by panel (non-CGN):
  panel     n_exp   

In [ ]:
# ── Cell 9 — Scrape domains + IDRs from biophysical pages ────────────────────
import re, json, urllib.request, pandas as pd

domain_rows = []
errors = []

for gene in all_genes:
    url = f"{rsa_base}/{gene}.html"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            html = r.read().decode("utf-8", errors="replace")
        m = re.search(r'const domains\s*=\s*(\[.*?\]);', html, re.DOTALL)
        if not m:
            raise ValueError("domains block not found")
        # NaN in JSON is invalid — replace before parsing
        domains_json = m.group(1).replace(": NaN", ": null")
        domains = json.loads(domains_json)
        for d in domains:
            domain_rows.append({
                "gene"       : gene,
                "panel"      : gene_panel[gene],
                "type"       : d.get("type"),
                "description": d.get("description"),
                "start"      : d.get("start"),
                "end"        : d.get("end"),
            })
    except Exception as e:
        errors.append(gene)
        print(f"  ERROR {gene}: {e}")

df_domains = pd.DataFrame(domain_rows)

# Summary of annotation types
print("Domain annotation types across all 129 genes:")
print(df_domains["type"].value_counts().to_string())
print(f"\nTotal annotation records : {len(df_domains):,}")
print(f"Genes with annotations   : {df_domains['gene'].nunique()}")
if errors:
    print(f"Errors: {errors}")

  ERROR MTTL1: HTTP Error 404: Not Found
Domain annotation types across all 129 genes:
type
Disulfide bond        563
Modified residue      386
Domain                380
Region                317
Compositional bias    305
Binding site          268
Transmembrane         180
Chain                 156
Motif                  63
Site                   46
Signal                 39
Coiled coil            37
Active site            24
Transit peptide         5
Propeptide              4

Total annotation records : 2,773
Genes with annotations   : 129
Errors: ['MTTL1']


In [ ]:
# ── Cell 9b — Annotate variants with domain + IDR ────────────────────────────

# Separate IDRs from other annotations
df_idr_regions = df_domains[
    (df_domains["type"] == "Region") &
    (df_domains["description"] == "Disordered")
].copy()

df_func_regions = df_domains[
    (df_domains["type"] == "Region") &
    (df_domains["description"] != "Disordered")
].copy()

# Keep functionally rich annotation types for domain mapping
domain_types = {"Domain", "Active site", "Binding site", "Motif",
                "Transmembrane", "Coiled coil", "Site", "Modified residue"}

df_anno = df_domains[df_domains["type"].isin(domain_types)].copy()

def annotate_variant(gene, resnum):
    """Return (domain_label, in_idr, in_functional_region) for a variant."""
    res = int(resnum)

    # IDR check
    idr = df_idr_regions[
        (df_idr_regions["gene"] == gene) &
        (df_idr_regions["start"] <= res) &
        (df_idr_regions["end"]   >= res)
    ]
    in_idr = len(idr) > 0

    # Functional region check
    func = df_func_regions[
        (df_func_regions["gene"] == gene) &
        (df_func_regions["start"] <= res) &
        (df_func_regions["end"]   >= res)
    ]
    func_label = func.iloc[0]["description"] if len(func) > 0 else None

    # Domain check (most specific: prioritize Active site > Binding site > Domain)
    priority = ["Active site", "Binding site", "Motif", "Site",
                "Transmembrane", "Coiled coil", "Domain"]
    domain_label = None
    for ptype in priority:
        hit = df_anno[
            (df_anno["gene"]  == gene) &
            (df_anno["type"]  == ptype) &
            (df_anno["start"] <= res) &
            (df_anno["end"]   >= res)
        ]
        if len(hit) > 0:
            domain_label = f"{ptype}: {hit.iloc[0]['description']}"
            break

    return domain_label, in_idr, func_label

# Apply to all variants — will take ~1 min
print("Annotating variants...")
results = df_rsa_clean.apply(
    lambda row: annotate_variant(row["gene"], row["resnum"]), axis=1
)
df_rsa_clean[["domain_label", "in_idr", "func_region"]] = pd.DataFrame(
    results.tolist(), index=df_rsa_clean.index
)

print(f"Done.")
print(f"\nDomain annotation rate : {df_rsa_clean['domain_label'].notna().sum():,} / {len(df_rsa_clean):,} ({100*df_rsa_clean['domain_label'].notna().mean():.1f}%)")
print(f"IDR annotation rate    : {df_rsa_clean['in_idr'].sum():,} / {len(df_rsa_clean):,} ({100*df_rsa_clean['in_idr'].mean():.1f}%)")
print(f"Func region rate       : {df_rsa_clean['func_region'].notna().sum():,} / {len(df_rsa_clean):,} ({100*df_rsa_clean['func_region'].notna().mean():.1f}%)")

print(f"\nTop domain labels in P/LP variants:")
plp = df_rsa_clean[df_rsa_clean["acmg"].isin(["P","LP"])]
print(plp["domain_label"].value_counts().head(15).to_string())

Annotating variants...
Done.

Domain annotation rate : 18,057 / 50,346 (35.9%)
IDR annotation rate    : 6,633 / 50,346 (13.2%)
Func region rate       : 9,479 / 50,346 (18.8%)

Top domain labels in P/LP variants:
domain_label
Transmembrane: Helical                 83
Domain: GBD/FH3                        54
Domain: Collagen IV NC1                49
Domain: NR LBD                         47
Domain: POU-specific atypical          39
Domain: D10C                           37
Coiled coil: None                      33
Binding site: None                     32
Domain: 5-PPase                        26
Domain: EGF-like 3; calcium-binding    23
Domain: Protein kinase                 23
Domain: Myosin motor                   21
Domain: Sushi 20                       17
Domain: SRCR                           16
Transmembrane: Helical; Name=3         13


In [ ]:
# ── Mutual exclusivity check — surface-exposed P/LP annotation overlap ────────

exposed_plp = df_rsa_clean[
    (df_rsa_clean["acmg"].isin(["P", "LP"])) &
    (df_rsa_clean["rsa"] >= 0.25) &
    (df_rsa_clean["panel"] != "CGN")
].copy()

has_domain = exposed_plp["domain_label"].notna()
has_idr    = exposed_plp["in_idr"]
has_func   = exposed_plp["func_region"].notna()

# All 7 combinations of 3 binary flags
combos = {
    "domain only"          : ( has_domain & ~has_idr & ~has_func).sum(),
    "IDR only"             : (~has_domain &  has_idr & ~has_func).sum(),
    "func only"            : (~has_domain & ~has_idr &  has_func).sum(),
    "domain + func"        : ( has_domain & ~has_idr &  has_func).sum(),
    "domain + IDR"         : ( has_domain &  has_idr & ~has_func).sum(),
    "IDR + func"           : (~has_domain &  has_idr &  has_func).sum(),
    "domain + IDR + func"  : ( has_domain &  has_idr &  has_func).sum(),
    "none"                 : (~has_domain & ~has_idr & ~has_func).sum(),
}

print(f"Surface-exposed P/LP (non-CGN): {len(exposed_plp)}")
print(f"\nMutually exclusive annotation categories:")
print(f"  {'category':<25} {'n':>5}  {'%':>6}")
print(f"  {'-'*38}")
for label, n in combos.items():
    print(f"  {label:<25} {n:>5}  {100*n/len(exposed_plp):>5.1f}%")

Surface-exposed P/LP (non-CGN): 769

Mutually exclusive annotation categories:
  category                      n       %
  --------------------------------------
  domain only                 196   25.5%
  IDR only                     27    3.5%
  func only                   109   14.2%
  domain + func                18    2.3%
  domain + IDR                  1    0.1%
  IDR + func                    0    0.0%
  domain + IDR + func           0    0.0%
  none                        418   54.4%


In [ ]:
# ── Supplemental summary table — domain breakdown ────────────────────────────

rows = []

# Class 1 — structural domains
for label, n in exposed_plp[has_domain & ~has_idr & ~has_func]["domain_label"].value_counts().items():
    rows.append({"annotation_class": "Structural domain", "annotation": label, "n": n})

# Class 2 — interaction interfaces
for label, n in exposed_plp[~has_domain & ~has_idr & has_func]["func_region"].value_counts().items():
    rows.append({"annotation_class": "Interaction interface", "annotation": label, "n": n})

# Class 3 — IDR
rows.append({"annotation_class": "IDR", "annotation": "Disordered region", "n": 25})

# Overlap
for label, n in exposed_plp[has_domain & ~has_idr & has_func]["domain_label"].value_counts().items():
    rows.append({"annotation_class": "Domain + interface overlap", "annotation": label, "n": n})

# Unannotated
rows.append({"annotation_class": "Unannotated", "annotation": "No UniProt feature", "n": 337})

df_supp_domains = pd.DataFrame(rows)
df_supp_domains.to_csv("supp_surface_exposed_plp_domains.csv", index=False)
print(f"Saved: supp_surface_exposed_plp_domains.csv ({len(df_supp_domains)} rows)")
print(df_supp_domains.to_string(index=False))

Saved: supp_surface_exposed_plp_domains.csv (91 rows)
          annotation_class                                                         annotation   n
         Structural domain                                                  Coiled coil: None  25
         Structural domain                                             Transmembrane: Helical  18
         Structural domain                                                     Domain: NR LBD  14
         Structural domain                                      Domain: POU-specific atypical  12
         Structural domain                                Domain: EGF-like 3; calcium-binding  11
         Structural domain                                                       Domain: D10C  11
         Structural domain                                                 Domain: EGF-like 4   7
         Structural domain                                             Domain: Protein kinase   6
         Structural domain                                      